In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:
from dotenv import load_dotenv
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client import models
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import torch
from tqdm import tqdm

In [3]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
load_dotenv(".env")

model_name = 'embedding_v3.1'
ratio = '80:10:10'
train_split = '80'
window_len = os.getenv("WINDOW_SIZE")
stride_len = os.getenv("STRIDE")
num_batch = os.getenv("BATCH_SIZE")
num_epoch = os.getenv("EPOCHS")
margin = 0.2
data_type = os.getenv("DATA_TYPE")  # Read from env, default to 'eo'
seeder = os.getenv("SEEDER")
wandb_name = model_name + "_" + str(data_type) + '_train_' + train_split + '_' + str(seeder) + '_' + str(window_len) + '_' + str(stride_len) + '_b' + str(num_batch) + '_e' + str(num_epoch) + '_margin_' + str(margin)
print(wandb_name)

embedding_v3.1_ec_train_80_42_2_2_b32_e100_margin_0.2


In [4]:
model = torch.load(f"models/{wandb_name}.pth", weights_only=False)

In [5]:
BASE_PATH = os.getenv("BASE_PATH")
PREPROCESSED_PATH = os.getenv("PREPROCESSED_PATH")
NPZ_PATH = os.getenv("NPZ_PATH")
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

suffix = f"{os.getenv('WINDOW_SIZE').replace('.', '')}_{os.getenv('STRIDE').replace('.', '')}"
preprocessed_dir = Path(BASE_PATH + PREPROCESSED_PATH)

X_train = np.load(preprocessed_dir / f'X_{data_type}_train_{suffix}.npy')
y_train = np.load(preprocessed_dir / f'y_{data_type}_train_{suffix}.npy')
X_val = np.load(preprocessed_dir / f'X_{data_type}_val_{suffix}.npy')
y_val = np.load(preprocessed_dir / f'y_{data_type}_val_{suffix}.npy')
X_test = np.load(preprocessed_dir / f'X_{data_type}_test_{suffix}.npy')
y_test = np.load(preprocessed_dir / f'y_{data_type}_test_{suffix}.npy')

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (2616, 64, 320) y_train: (2616,)
X_val: (327, 64, 320) y_val: (327,)
X_test: (327, 64, 320) y_test: (327,)


In [6]:
X_train_t = torch.from_numpy(X_train.copy()).float()
y_train_t = torch.from_numpy(y_train.copy()).long()

X_test_t = torch.from_numpy(X_test.copy()).float()
y_test_t = torch.from_numpy(y_test.copy()).long()

X_val_t = torch.from_numpy(X_val.copy()).float()
y_val_t = torch.from_numpy(y_val.copy()).long()

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(
    train_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=True,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=True
)
val_loader = DataLoader(
    val_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False
)
test_loader = DataLoader(
    test_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False
)

In [7]:
# Enrollment gallery: train + val (test held out for evaluation)

client = QdrantClient(url="http://localhost:6333")

model.eval()
X_enroll = np.concatenate([X_train, X_val], axis=0)
y_enroll = np.concatenate([y_train, y_val], axis=0)
collection_name = wandb_name + "_euclidean"

emb_batch = int(os.getenv("BATCH_SIZE"))
enroll_ds = TensorDataset(
    torch.from_numpy(X_enroll).float(),
    torch.from_numpy(y_enroll).long(),
)
enroll_loader = DataLoader(
    enroll_ds,
    batch_size=emb_batch,
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False,
)

emb_chunks, label_chunks = [], []
with torch.no_grad():
    for data_eeg, targets in tqdm(enroll_loader, desc="Extract embeddings (enrollment)"):
        data_eeg = data_eeg.to("cuda", non_blocking=True)
        emb = model(data_eeg).cpu().numpy()
        emb_chunks.append(emb)
        label_chunks.append(targets.numpy())

embeddings_matrix = np.concatenate(emb_chunks, axis=0)
subject_ids = np.concatenate(label_chunks, axis=0)

out_path = Path(BASE_PATH + NPZ_PATH) / f"{wandb_name}_euclidean.npz"
out_path.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(out_path, embeddings=embeddings_matrix, subject_ids=subject_ids)
print(f"Saved local embedding backup: {out_path}  shape={embeddings_matrix.shape}")

qdrant_batch = 256
for start in tqdm(
    range(0, len(embeddings_matrix), qdrant_batch),
    desc="Upsert to Qdrant",
):
    end = min(start + qdrant_batch, len(embeddings_matrix))
    points = [
        models.PointStruct(
            id=start + i,
            vector=embeddings_matrix[start + i].tolist(),
            payload={"subject_id": int(subject_ids[start + i])},
        )
        for i in range(end - start)
    ]

    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(size=128, distance=models.Distance.EUCLID),
        )
    client.upsert(collection_name=collection_name, points=points)

print(f"Upserted {len(embeddings_matrix)} points to collection {collection_name}.")

Extract embeddings (enrollment):   0%|                                                                                                                  | 0/92 [00:00<?, ?it/s]

/home/chocomaltt/Kuliah/eeg-biometric-system/eeg/lib/python3.10/site-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv2d(
Extract embeddings (enrollment):   1%|█▏                                                                                                        | 1/92 [00:00<00:25,  3.54it/s]

Extract embeddings (enrollment):   5%|█████▊                                                                                                    | 5/92 [00:00<00:05, 14.91it/s]

Extract embeddings (enrollment):  10%|██████████▎                                                                                               | 9/92 [00:00<00:03, 22.08it/s]

Extract embeddings (enrollment):  14%|██████████████▊                                                                                          | 13/92 [00:00<00:02, 26.95it/s]

...

Upsert to Qdrant:  58%|██████████████████████████████████████████████████████████████████████▌                                                  | 7/12 [00:00<00:00, 10.93it/s]

Upsert to Qdrant:  75%|██████████████████████████████████████████████████████████████████████████████████████████▊                              | 9/12 [00:00<00:00, 11.90it/s]

Upsert to Qdrant:  92%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████          | 11/12 [00:01<00:00, 12.19it/s]

Upsert to Qdrant: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:01<00:00, 10.35it/s]

Upserted 2943 points to collection embedding_v3.1_ec_train_80_42_2_2_b32_e100_margin_0.2_euclidean.


In [8]:
# === EUCLIDEAN DISTANCE EVALUATION (ROC/EER) ===
# Note: For Euclidean distance, LOWER scores = MORE similar (opposite of cosine)

from sklearn.metrics import roc_curve

roc_query_limit_euc = 200

y_true_euc = []
y_scores_euc = []
top1_correct_euc = 0
total_test_samples_euc = 0

with torch.no_grad():
    for data_eeg, targets in tqdm(test_loader, desc="Euclidean ROC Evaluation"):
        data_eeg = data_eeg.to("cuda", non_blocking=True)
        embeddings = model(data_eeg).cpu().numpy()
        targets = targets.numpy()

        for i in range(len(embeddings)):
            query_vector = embeddings[i].tolist()
            true_label = int(targets[i])

            search_result = client.query_points(
                collection_name=collection_name,
                query=query_vector,
                limit=roc_query_limit_euc
            )
            points = search_result.points
            if not points:
                continue

            best_match = points[0]
            predicted_label = int(best_match.payload["subject_id"])
            top1_correct_euc += int(predicted_label == true_label)
            total_test_samples_euc += 1

            for point in points:
                candidate_label = int(point.payload["subject_id"])
                # For Euclidean: convert distance to similarity (negate for ROC)
                # Lower distance = more similar, so we negate for consistent ROC calculation
                y_true_euc.append(1 if candidate_label == true_label else 0)
                y_scores_euc.append(-point.score)  # Negate so higher = more similar

y_true_euc = np.array(y_true_euc)
y_scores_euc = np.array(y_scores_euc)

classes_euc, counts_euc = np.unique(y_true_euc, return_counts=True)
class_counts_euc = dict(zip(classes_euc.tolist(), counts_euc.tolist()))
print("Euclidean ROC label counts:", class_counts_euc)

fpr_euc, tpr_euc, thresholds_euc = roc_curve(y_true_euc, y_scores_euc)
far_euc = fpr_euc
frr_euc = 1 - tpr_euc

eer_idx_euc = np.nanargmin(np.abs(far_euc - frr_euc))
eer_euc = (far_euc[eer_idx_euc] + frr_euc[eer_idx_euc]) / 2
eer_threshold_euc = -thresholds_euc[eer_idx_euc]  # Convert back to positive distance
top1_accuracy_euc = top1_correct_euc / total_test_samples_euc

print("\n=== EUCLIDEAN DISTANCE EVALUATION RESULTS ===")
print(f"Total Test Samples : {total_test_samples_euc}")
print(f"Total ROC Scores   : {len(y_true_euc)}")
print(f"Genuine / Impostor : {class_counts_euc.get(1, 0)} / {class_counts_euc.get(0, 0)}")
print(f"Top-1 Accuracy     : {top1_accuracy_euc * 100:.2f}%")
print(f"EER                : {eer_euc * 100:.2f}%")
print(f"EER Threshold (distance) : {eer_threshold_euc:.4f}")

Euclidean ROC Evaluation:   0%|                                                                                                                         | 0/11 [00:00<?, ?it/s]

Euclidean ROC Evaluation:   9%|██████████▎                                                                                                      | 1/11 [00:00<00:02,  3.80it/s]

Euclidean ROC Evaluation:  18%|████████████████████▌                                                                                            | 2/11 [00:00<00:01,  4.98it/s]

Euclidean ROC Evaluation:  27%|██████████████████████████████▊                                                                                  | 3/11 [00:00<00:01,  5.22it/s]

Euclidean ROC Evaluation:  36%|█████████████████████████████████████████                                                                        | 4/11 [00:00<00:01,  5.59it/s]

...

Euclidean ROC Evaluation:  73%|██████████████████████████████████████████████████████████████████████████████████▏                              | 8/11 [00:01<00:00,  5.96it/s]

Euclidean ROC Evaluation:  82%|████████████████████████████████████████████████████████████████████████████████████████████▍                    | 9/11 [00:01<00:00,  5.94it/s]

Euclidean ROC Evaluation:  91%|█████████████████████████████████████████████████████████████████████████████████████████████████████▊          | 10/11 [00:01<00:00,  6.04it/s]

Euclidean ROC Evaluation: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:01<00:00,  5.96it/s]

Euclidean ROC label counts: {0: 56690, 1: 8710}

=== EUCLIDEAN DISTANCE EVALUATION RESULTS ===
Total Test Samples : 327
Total ROC Scores   : 65400
Genuine / Impostor : 8710 / 56690
Top-1 Accuracy     : 98.78%
EER                : 5.29%
EER Threshold (distance) : 0.5223
